In [ ]:
# Resolve public-release paths without requiring internal project documentation.
from pathlib import Path as _ReleasePath
import os as _release_os
_release_root = next((p for p in (_ReleasePath.cwd(), *_ReleasePath.cwd().parents)
                      if (p / ".shap_multiplicity_root").is_file()), None)
if _release_root is None:
    raise FileNotFoundError("Launch Jupyter within this repository.")
_release_os.chdir(_release_root)

import os, pickle, itertools, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pairwise_aggregation import pairwise_distances, pairwise_exceedance
from IPython.display import display

from plot_violin import (
    compute_rbo_distance, compute_jaccard_distance, compute_l2_distance,
    compute_rbo_baseline, compute_jaccard_baseline, compute_l2_baseline,
)

DATASETS   = ['german', 'diabetes', 'acs', 'gmsc']
MODELS     = ['dt', 'rf', 'xgb', 'mlp', 'ftt']
EXPLAINERS = ['kernel_shap', 'permutation_shap']
METHODS    = ['random', 'kmeans', 'pred_stratified', 'cte']
METRICS    = ['l2', 'jac', 'rbo']

DATASET_LABELS = {'german': 'German', 'diabetes': 'Diabetes', 'acs': 'ACS', 'gmsc': 'GMSC'}
MODEL_LABELS   = {'dt': 'DT', 'rf': 'RF', 'xgb': 'XGB', 'mlp': 'MLP', 'ftt': 'FTT'}
EXPL_LABELS    = {'kernel_shap': 'KernelSHAP', 'permutation_shap': 'PermSHAP'}
METHOD_LABELS  = {'random': 'Random', 'kmeans': 'KMeans',
                  'pred_stratified': 'Pred-Strat', 'cte': 'CTE'}
METRIC_LABELS  = {'l2': 'L2', 'jac': 'Top-3 Jaccard Distance', 'rbo': '1-RBO'}

N_BG_SEEDS = 5
N_CHUNKS   = {'german': 1, 'diabetes': 1, 'acs': 100, 'gmsc': 100}
N_FEATS    = {'german': 16, 'diabetes': 8, 'acs': 8, 'gmsc': 10, 'arrhythmia': 278}
JAC_K      = 3
RES_DIR    = 'results_tree'
TABLE_DIR  = 'figures_b/background_table'
os.makedirs(TABLE_DIR, exist_ok=True)
PAIRS_5    = list(itertools.combinations(range(5), 2))  # 10 BG seed pairs

print('Constants loaded.')

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

def load_sv(ds, model, model_seed, method, s_idx, explainer='kernel_shap'):
    prefix = 'perm_' if explainer == 'permutation_shap' else ''
    n_ch   = N_CHUNKS[ds]

    def _chunk(c=None):
        base = f'{ds}_{model}_0_{model_seed}_{prefix}{method}_s{s_idx}'
        fn   = f'{base}_c{c}_sv.pkl' if c is not None else f'{base}_sv.pkl'
        p    = os.path.join(RES_DIR, fn)
        if not os.path.exists(p): return None
        with open(p, 'rb') as f: obj = pickle.load(f)
        return obj.values

    if n_ch == 1:
        return _chunk()
    arrs = [_chunk(c) for c in range(n_ch)]
    if any(a is None for a in arrs):
        print(f'[INCOMPLETE] {ds}/{model}/{method}/seed{s_idx}: expected {n_ch} chunks')
        return None
    return np.concatenate(arrs, axis=0)


def pct_above(dist_arrays, threshold):
    """B: strict exceedances divided by 10*N; never mean distance per instance."""
    return pairwise_exceedance(dist_arrays, threshold)['percent']


def dist_fn(metric, v1, v2):
    if metric == 'rbo': return compute_rbo_distance(v1, v2)
    if metric == 'jac': return compute_jaccard_distance(v1, v2, k=JAC_K)
    if metric == 'l2':  return compute_l2_distance(v1, v2)


print('Helpers ready.')

In [ ]:
# ── Baselines per dataset ─────────────────────────────────────────────────────

baselines = {}
for ds in DATASETS:
    d = N_FEATS[ds]
    r = max(1, d // 3)
    baselines[ds] = {
        'rbo': compute_rbo_baseline(n_features=d)['min'],
        'jac': compute_jaccard_baseline(d=d, r=JAC_K)['min'],
        'l2':  compute_l2_baseline(d=d, r=r)['min'],
    }
    print(f"{ds} (d={d}): "
          f"rbo>{baselines[ds]['rbo']:.4f}  "
          f"jac>{baselines[ds]['jac']:.4f}  "
          f"l2>{baselines[ds]['l2']:.4f}")

In [ ]:
# B: each cell reports exceedances / (10 * N), with complete five-run inputs.
# Missing/mismatched groups remain NaN and are reported, never silently trimmed.
table_data = {
    mt: {ds: {model: {expl: {} for expl in EXPLAINERS} for model in MODELS}
         for ds in DATASETS} for mt in METRICS
}
aggregation_records = []
for ds in DATASETS:
    for model in MODELS:
        for expl in EXPLAINERS:
            for method in METHODS:
                svs = [load_sv(ds, model, 0, method, s, expl) for s in range(N_BG_SEEDS)]
                for mt in METRICS:
                    record = dict(dataset=ds, model=model, explainer=expl, method=method, metric=mt,
                                  input_layout='chunks' if N_CHUNKS[ds] > 1 else 'whole')
                    try:
                        distances = pairwise_distances(svs, lambda a, b: dist_fn(mt, a, b))
                        stats = pairwise_exceedance(distances, baselines[ds][mt])
                        table_data[mt][ds][model][expl][method] = stats['percent']
                        record.update(status='complete', **stats)
                    except ValueError as error:
                        table_data[mt][ds][model][expl][method] = float('nan')
                        record.update(status='not_computed', reason=str(error))
                        print(f'[NOT COMPUTED] {ds}/{model}/{expl}/{method}/{mt}: {error}')
                    aggregation_records.append(record)
        print(f'{ds}/{model}: done')
pd.DataFrame(aggregation_records).to_csv(os.path.join(TABLE_DIR, 'pair_exceedance_counts.csv'), index=False)
print('B aggregation complete; numerator, 10N denominator, and missing groups recorded.')


In [ ]:
# ── Heatmaps: one figure per (metric, explainer) ─────────────────────────────

import seaborn as sns
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science', 'no-latex'])
import matplotlib
matplotlib.rcParams.update({
    'font.size':        13,
    'axes.labelsize':   14,
    'xtick.labelsize':  12,
    'ytick.labelsize':  12,
    'legend.fontsize':  12,
})

os.makedirs(TABLE_DIR, exist_ok=True)

ROW_LABELS = [f'{DATASET_LABELS[ds]} / {MODEL_LABELS[m]}'
              for ds in DATASETS for m in MODELS]
COL_LABELS = [METHOD_LABELS[m] for m in METHODS]
row_bounds = [k * len(MODELS) for k in range(1, len(DATASETS))]

def make_annot(mat, fmt):
    """Custom annotation: empty string for NaN, formatted number otherwise."""
    out = np.empty(mat.shape, dtype=object)
    for idx in np.ndindex(mat.shape):
        v = mat[idx]
        out[idx] = '' if np.isnan(v) else format(v, fmt)
    return out

for mt in METRICS:
    for expl in EXPLAINERS:
        fig, ax = plt.subplots(figsize=(6, 8))

        mat = np.full((len(DATASETS) * len(MODELS), len(METHODS)), float('nan'))
        for i, ds in enumerate(DATASETS):
            for j, model in enumerate(MODELS):
                for l, method in enumerate(METHODS):
                    mat[i * len(MODELS) + j, l] = \
                        table_data[mt][ds][model][expl][method]

        df_heat = pd.DataFrame(mat, index=ROW_LABELS, columns=COL_LABELS)

        sns.heatmap(df_heat, ax=ax, vmin=0, vmax=100, cmap='YlOrRd',
                    annot=make_annot(mat, '.0f'), fmt='', annot_kws={'size': 11},
                    linewidths=0.3, linecolor='white',
                    cbar_kws={'label': '% comparisons > baseline', 'shrink': 0.6})

        for rb in row_bounds:
            ax.axhline(rb, color='black', linewidth=1.5)

        ax.set_xlabel('Sampling Method', fontsize=14)
        ax.tick_params(axis='x', rotation=30, labelsize=12)
        ax.tick_params(axis='y', rotation=0,  labelsize=12)

        fig.tight_layout()
        fname = os.path.join(TABLE_DIR, f'bg_heatmap_{mt}_{expl}.pdf')
        fig.savefig(fname, bbox_inches='tight')
        plt.show(); plt.close(fig)
        print(f'Saved: {fname}')

In [ ]:
# ── GMSC-only heatmaps, kernel_shap only (for paper) ──────────────────────────
DS_GMSC = 'gmsc'
EXPL_PAPER = 'kernel_shap'
gmsc_row_labels = [f'{DATASET_LABELS[DS_GMSC]} / {MODEL_LABELS[m]}' for m in MODELS]

for mt in METRICS:
    mat_gmsc = np.full((len(MODELS), len(METHODS)), float('nan'))
    for j, model in enumerate(MODELS):
        for l, method in enumerate(METHODS):
            mat_gmsc[j, l] = table_data[mt][DS_GMSC][model][EXPL_PAPER][method]

    df_gmsc = pd.DataFrame(mat_gmsc, index=gmsc_row_labels, columns=COL_LABELS)

    fig, ax = plt.subplots(figsize=(6, 3))
    sns.heatmap(df_gmsc, ax=ax, vmin=0, vmax=100, cmap='YlOrRd',
                annot=make_annot(mat_gmsc, '.0f'), fmt='', annot_kws={'size': 11},
                linewidths=0.3, linecolor='white',
                cbar_kws={'label': '% comparisons > baseline', 'shrink': 0.6})

    ax.set_xlabel('Sampling Method', fontsize=14)
    ax.tick_params(axis='x', rotation=30, labelsize=12)
    ax.tick_params(axis='y', rotation=0, labelsize=12)

    fig.tight_layout()
    fname = os.path.join(TABLE_DIR, f'bg_heatmap_{mt}_{EXPL_PAPER}_gmsc.pdf')
    fig.savefig(fname, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')